[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ssuai/Hands-On-Large-Language-Models/blob/main/chapter02/lab_2_1_tokenization.ipynb)

<h1>Lab 2-1 Tokenization</h1>

### [OPTIONAL] - Installing Packages on Colab

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [1]:
%%capture
!pip install --upgrade transformers==4.41.2 sentence-transformers==3.0.1 gensim==4.3.2 scikit-learn==1.5.0 accelerate==0.31.0 peft==0.11.1 scipy==1.10.1 numpy==1.26.4

# Comparing Trained LLM Tokenizers


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

In [3]:
def analyze_tab_tokenization(model_name):
    """
    Analyzes how a specific model's tokenizer handles sequences of tabs (\t).

    Args:
        model_name (str): The Hugging Face model identifier.
    """
    try:
        # Load tokenizer (trust_remote_code is often needed for newer models like Phi-3)
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

        # Prepare test cases: 1 to 4 tabs
        test_cases = [f"{'\t' * i}" for i in range(1, 5)]

        print(f"{'Input (Tabs)':<15} | {'Tokens':<25} | {'Token IDs':<20}")
        print("-" * 65)

        for text in test_cases:
            # Tokenize and encode
            tokens = tokenizer.tokenize(text)
            input_ids = tokenizer.encode(text, add_special_tokens=False)

            # Visualization: Replace tab with [T] to see clearly
            visual_input = f"{len(text)} Tab(s) " + ("[T]" * len(text))

            print(f"{visual_input:<15} | {str(tokens):<25} | {str(input_ids):<20}")

    except Exception as e:
        print(f"Error loading model '{model_name}': {e}")

In [4]:
text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"\t\t" Three tabs: "\t\t\t"
12.0*50=600
"""

In [5]:
show_tokens(text, "bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[CLS] english and capital ##ization [UNK] [UNK] show _ token ##s false none eli ##f = = > = else : two tab ##s : " " three tab ##s : " " 12 . 0 * 50 = 600 [SEP] 

In [6]:
analyze_tab_tokenization("bert-base-uncased")

Input (Tabs)    | Tokens                    | Token IDs           
-----------------------------------------------------------------
1 Tab(s) [T]    | []                        | []                  
2 Tab(s) [T][T] | []                        | []                  
3 Tab(s) [T][T][T] | []                        | []                  
4 Tab(s) [T][T][T][T] | []                        | []                  


In [7]:
show_tokens(text, "bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[CLS] English and CA ##PI ##TA ##L ##I ##Z ##AT ##ION [UNK] [UNK] show _ token ##s F ##als ##e None el ##if = = > = else : two ta ##bs : " " Three ta ##bs : " " 12 . 0 * 50 = 600 [SEP] 

In [8]:
analyze_tab_tokenization("bert-base-cased")

Input (Tabs)    | Tokens                    | Token IDs           
-----------------------------------------------------------------
1 Tab(s) [T]    | []                        | []                  
2 Tab(s) [T][T] | []                        | []                  
3 Tab(s) [T][T][T] | []                        | []                  
4 Tab(s) [T][T][T][T] | []                        | []                  


In [9]:
show_tokens(text, "gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


 English  and  CAP ITAL IZ ATION 
 � � �  � � � 
 show _ t ok ens  False  None  el if  ==  >=  else :  two  tabs :" 	 	 "  Three  tabs :  " 	 	 	 " 
 12 . 0 * 50 = 600 
 

In [10]:
analyze_tab_tokenization("gpt2")

Input (Tabs)    | Tokens                    | Token IDs           
-----------------------------------------------------------------
1 Tab(s) [T]    | ['ĉ']                     | [197]               
2 Tab(s) [T][T] | ['ĉ', 'ĉ']                | [197, 197]          
3 Tab(s) [T][T][T] | ['ĉ', 'ĉ', 'ĉ']           | [197, 197, 197]     
4 Tab(s) [T][T][T][T] | ['ĉ', 'ĉ', 'ĉ', 'ĉ']      | [197, 197, 197, 197]


In [11]:
show_tokens(text, "google/flan-t5-small")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

English and CA PI TAL IZ ATION  <unk>  <unk> show _ to ken s Fal s e None  e l if = = > = else : two tab s : " " Three tab s : " " 12. 0 * 50 = 600 </s> 

In [12]:
# The official is `tiktoken` but this the same tokenizer on the HF platform
show_tokens(text, "Xenova/gpt-4")

tokenizer_config.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


 English  and  CAPITAL IZATION 
 � � �  � � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :" 	 	 "  Three  tabs :  " 		 	 " 
 12 . 0 * 50 = 600 
 

In [13]:
analyze_tab_tokenization("Xenova/gpt-4")

Input (Tabs)    | Tokens                    | Token IDs           
-----------------------------------------------------------------
1 Tab(s) [T]    | ['ĉ']                     | [197]               
2 Tab(s) [T][T] | ['ĉĉ']                    | [298]               
3 Tab(s) [T][T][T] | ['ĉĉĉ']                   | [573]               
4 Tab(s) [T][T][T][T] | ['ĉĉĉĉ']                  | [465]               


In [14]:
# You need to request access before being able to use this tokenizer
show_tokens(text, "bigcode/starcoder2-15b")

config.json:   0%|          | 0.00/803 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


 English  and  CAPITAL IZATION 
 � � �   � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :" 	 	 "  Three  tabs :  " 		 	 " 
 1 2 . 0 * 5 0 = 6 0 0 
 

In [15]:
# --- Test with Galactica ---
analyze_tab_tokenization("bigcode/starcoder2-15b")

Input (Tabs)    | Tokens                    | Token IDs           
-----------------------------------------------------------------
1 Tab(s) [T]    | ['ĉ']                     | [221]               
2 Tab(s) [T][T] | ['ĉĉ']                    | [313]               
3 Tab(s) [T][T][T] | ['ĉĉĉ']                   | [3177]              
4 Tab(s) [T][T][T][T] | ['ĉĉĉĉ']                  | [1017]              


In [16]:
show_tokens(text, "facebook/galactica-1.3b")

config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.00 [00:00<?, ?B/s]


 English  and  CAP ITAL IZATION 
 � � � �  � � � 
 show _ tokens  False  None  elif   ==   > =  else :  two  t abs : " 		 "  Three  t abs :   " 			 " 
 1 2 . 0 * 5 0 = 6 0 0 
 

In [17]:
# --- Test with Galactica ---
analyze_tab_tokenization("facebook/galactica-1.3b")

Input (Tabs)    | Tokens                    | Token IDs           
-----------------------------------------------------------------
1 Tab(s) [T]    | ['ĉ']                     | [220]               
2 Tab(s) [T][T] | ['ĉĉ']                    | [1211]              
3 Tab(s) [T][T][T] | ['ĉĉĉ']                   | [4388]              
4 Tab(s) [T][T][T][T] | ['ĉĉĉĉ']                  | [5326]              


In [18]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

 
 English and C AP IT AL IZ ATION 
 � � � �  � � � 
 show _ to kens False None elif == >= else : two tabs :" 	 	 " Three tabs : " 	 	 	 " 
 1 2 . 0 * 5 0 = 6 0 0 
 

In [19]:
analyze_tab_tokenization("microsoft/Phi-3-mini-4k-instruct")

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Input (Tabs)    | Tokens                    | Token IDs           
-----------------------------------------------------------------
1 Tab(s) [T]    | ['▁', '<0x09>']           | [29871, 12]         
2 Tab(s) [T][T] | ['▁', '<0x09>', '<0x09>'] | [29871, 12, 12]     
3 Tab(s) [T][T][T] | ['▁', '<0x09>', '<0x09>', '<0x09>'] | [29871, 12, 12, 12] 
4 Tab(s) [T][T][T][T] | ['▁', '<0x09>', '<0x09>', '<0x09>', '<0x09>'] | [29871, 12, 12, 12, 12]
